# TLA⁺ in your browser

Nothing is installed on your machine. No Java, no `tla2tools.jar`, no account.

This page runs Python in your own browser tab. When you check a specification, the spec text is sent to a small public service that runs TLC and returns the result as JSON — everything you see below is rendered here, from that JSON.

Run the cells in order (`Shift`+`Enter`).

In [ ]:
%pip install -q tlakit

In [ ]:
%load_ext tlakit

That one line did three things, because in a browser there is no other way to work: it registered the `%%tla` and `%%tlc` magics, pointed checking at the remote runner, and turned on cell routing — so a cell that *starts with a module header is TLA⁺*, with no magic needed.

## A spec with a real bug

Two threads each increment a shared counter, but reading and writing are separate steps. Both can read `0` before either writes.

In [ ]:
---- MODULE LostUpdate ----
EXTENDS Naturals
CONSTANT Threads
VARIABLES counter, tmp, pc

Init ==
  /\ counter = 0
  /\ tmp = [t \in Threads |-> 0]
  /\ pc = [t \in Threads |-> "read"]

Read(t) ==
  /\ pc[t] = "read"
  /\ tmp' = [tmp EXCEPT ![t] = counter]
  /\ pc' = [pc EXCEPT ![t] = "write"]
  /\ UNCHANGED counter

Write(t) ==
  /\ pc[t] = "write"
  /\ counter' = tmp[t] + 1
  /\ pc' = [pc EXCEPT ![t] = "done"]
  /\ UNCHANGED tmp

Next == \E t \in Threads : Read(t) \/ Write(t)
Done == \A t \in Threads : pc[t] = "done"

Spec == Init /\ [][Next]_<<counter, tmp, pc>>

\* Every thread incremented once, so the counter should equal the thread count.
Correct == Done => counter = Cardinality(Threads)
====


Now the configuration. This cell has no magic either — TLC config keywords are recognised on their own.

In [ ]:
SPECIFICATION Spec
CONSTANT Threads = {"a", "b"}
INVARIANT Correct


The counter ends at `1`, not `2`. The trace above is the shortest way to get there: both threads read `0`, then both write `1`.

## The result is a Python object

This is the part a static tool cannot give you — the counterexample is data you can query.

In [ ]:
import tlakit
from tlakit.magics import MODULES

spec = tlakit.Spec(source=MODULES["LostUpdate"], name="LostUpdate")
result = spec.check(
    constants={"Threads": {"a", "b"}},
    invariants=["Correct"],
)

print("outcome:", result.outcome.value)
print("states explored:", result.stats.distinct)
print("counterexample length:", len(result.trace))
print("actions taken:", [a.name for a in result.trace.actions])
print("final counter:", result.trace.states[-1]["counter"])

In [ ]:
# Which variables changed at each step?
for i in range(len(result.trace)):
    print(i, sorted(result.trace.delta(i)) or "(initial)")

## Fix it and check again

Make the increment atomic — one action that reads and writes together — and the invariant holds.

In [ ]:
---- MODULE Atomic ----
EXTENDS Naturals, FiniteSets
CONSTANT Threads
VARIABLES counter, pc

Init ==
  /\ counter = 0
  /\ pc = [t \in Threads |-> "todo"]

Inc(t) ==
  /\ pc[t] = "todo"
  /\ counter' = counter + 1
  /\ pc' = [pc EXCEPT ![t] = "done"]

Next == \E t \in Threads : Inc(t)
Done == \A t \in Threads : pc[t] = "done"
Spec == Init /\ [][Next]_<<counter, pc>>
Correct == Done => counter = Cardinality(Threads)
====


In [ ]:
%tlc:Atomic
SPECIFICATION Spec
CONSTANT Threads = {"a", "b"}
INVARIANT Correct


## Your turn

Edit any cell above and re-run it. A few things to try:

- Add a third thread: `CONSTANT Threads = {"a", "b", "c"}`. The state space grows fast — that growth is the thing model checking is fighting.
- Break `Atomic` on purpose by dropping the `pc[t] = "todo"` guard, and see what TLC finds.
- Ask for the state graph: `spec.check(..., graph=True)` and then `result.graph`.

### Limits

The public runner allows 6 checks a minute, caps each at 30 seconds, and runs specs without CommunityModules — so a spec has no way to touch the filesystem or network. For anything larger, install tlakit locally with `pip install tlakit` and it will drive your own TLC with no limits at all.